
# Census Income Classification using PyTorch

This notebook builds a binary classification model using PyTorch to predict whether an individual earns more than $50,000 annually using the Census Income Dataset.



## Objectives
- Prepare categorical and continuous features
- Create embeddings for categorical columns
- Build a PyTorch tabular neural network
- Train the model using CrossEntropyLoss and Adam optimizer
- Evaluate model performance
- Predict income for new user inputs


In [1]:

# Import libraries
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Load dataset
df = pd.read_csv("income.csv")

print(df.shape)
df.head()


(30000, 10)


,age,sex,education,education-num,marital-status,workclass,occupation,hours-per-week,income,label
0,27,Male,HS-grad,9,Never-married,Private,Craft-repair,40,<=50K,0
1,47,Male,Masters,14,Married,Local-gov,Exec-managerial,50,>50K,1
2,59,Male,HS-grad,9,Divorced,Self-emp,Prof-specialty,20,<=50K,0
3,38,Female,Prof-school,15,Never-married,Federal-gov,Prof-specialty,57,>50K,1
4,64,Female,11th,7,Widowed,Private,Farming-fishing,40,<=50K,0


In [3]:

# Dataset information
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             30000 non-null  int64 
 1   sex             30000 non-null  object
 2   education       30000 non-null  object
 3   education-num   30000 non-null  int64 
 4   marital-status  30000 non-null  object
 5   workclass       30000 non-null  object
 6   occupation      30000 non-null  object
 7   hours-per-week  30000 non-null  int64 
 8   income          30000 non-null  object
 9   label           30000 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 2.3+ MB



## Data Preparation
We separate:
- Categorical columns
- Continuous columns
- Label column


In [4]:

# Define columns
cat_cols = ['sex', 'education', 'marital-status', 'workclass', 'occupation']
cont_cols = ['age', 'education-num', 'hours-per-week']
label_col = 'label'

print("Categorical Columns:", cat_cols)
print("Continuous Columns:", cont_cols)


Categorical Columns: ['sex', 'education', 'marital-status', 'workclass', 'occupation']
Continuous Columns: ['age', 'education-num', 'hours-per-week']


In [5]:

# Convert categorical columns to category codes
for col in cat_cols:
    df[col] = df[col].astype('category')

# Create categorical arrays
cat_sizes = [len(df[col].cat.categories) for col in cat_cols]

cat_emb_sizes = [(size, min(50, (size + 1) // 2)) for size in cat_sizes]

cats = np.stack([df[col].cat.codes.values for col in cat_cols], axis=1)

# Continuous values
conts = np.stack([df[col].values for col in cont_cols], axis=1).astype(np.float32)

# Labels
y = df[label_col].values

print("Categorical tensor shape:", cats.shape)
print("Continuous tensor shape:", conts.shape)
print("Labels shape:", y.shape)


Categorical tensor shape: (30000, 5)
Continuous tensor shape: (30000, 3)
Labels shape: (30000,)


In [6]:

# Convert to tensors
cats = torch.tensor(cats, dtype=torch.int64)
conts = torch.tensor(conts, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

print(cats.shape)
print(conts.shape)
print(y.shape)


torch.Size([30000, 5])
torch.Size([30000, 3])
torch.Size([30000])



## Train-Test Split
Use:
- 25,000 samples for training
- 5,000 samples for testing


In [7]:

# Split dataset
cat_train = cats[:25000]
cat_test = cats[25000:]

cont_train = conts[:25000]
cont_test = conts[25000:]

y_train = y[:25000]
y_test = y[25000:]

print("Training size:", len(y_train))
print("Testing size:", len(y_test))


Training size: 25000
Testing size: 5000


In [8]:

# Create DataLoader
batch_size = 64

train_data = TensorDataset(cat_train, cont_train, y_train)
test_data = TensorDataset(cat_test, cont_test, y_test)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size)



## Model Design
We build a TabularModel with:
- Embeddings for categorical features
- Batch normalization for continuous features
- One hidden layer with 50 neurons
- Dropout = 0.4


In [9]:

class TabularModel(nn.Module):

    def __init__(self, emb_sizes, n_cont, out_sz, layers, p=0.4):

        super().__init__()

        # Embedding layers
        self.embeds = nn.ModuleList([
            nn.Embedding(ni, nf) for ni, nf in emb_sizes
        ])

        self.emb_drop = nn.Dropout(p)

        # Batch normalization for continuous features
        self.bn_cont = nn.BatchNorm1d(n_cont)

        n_emb = sum([nf for ni, nf in emb_sizes])

        layerlist = []

        n_in = n_emb + n_cont

        for i in layers:
            layerlist.append(nn.Linear(n_in, i))
            layerlist.append(nn.ReLU(inplace=True))
            layerlist.append(nn.BatchNorm1d(i))
            layerlist.append(nn.Dropout(p))

            n_in = i

        layerlist.append(nn.Linear(layers[-1], out_sz))

        self.layers = nn.Sequential(*layerlist)

    def forward(self, x_cat, x_cont):

        embeddings = []

        for i, e in enumerate(self.embeds):
            embeddings.append(e(x_cat[:, i]))

        x = torch.cat(embeddings, 1)

        x = self.emb_drop(x)

        x_cont = self.bn_cont(x_cont)

        x = torch.cat([x, x_cont], 1)

        x = self.layers(x)

        return x


In [10]:

# Create model
model = TabularModel(
    emb_sizes=cat_emb_sizes,
    n_cont=len(cont_cols),
    out_sz=2,
    layers=[50],
    p=0.4
)

print(model)


TabularModel(
  (embeds): ModuleList(
    (0): Embedding(2, 1)
    (1): Embedding(14, 7)
    (2): Embedding(6, 3)
    (3): Embedding(5, 3)
    (4): Embedding(12, 6)
  )
  (emb_drop): Dropout(p=0.4, inplace=False)
  (bn_cont): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layers): Sequential(
    (0): Linear(in_features=23, out_features=50, bias=True)
    (1): ReLU(inplace=True)
    (2): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=50, out_features=2, bias=True)
  )
)



## Training
- Criterion: CrossEntropyLoss
- Optimizer: Adam
- Learning Rate: 0.001
- Epochs: 300


In [11]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [12]:

epochs = 300

for i in range(epochs):

    model.train()

    for cat_batch, cont_batch, y_batch in train_loader:

        optimizer.zero_grad()

        y_pred = model(cat_batch, cont_batch)

        loss = criterion(y_pred, y_batch)

        loss.backward()

        optimizer.step()

    if (i + 1) % 25 == 0:
        print(f"Epoch {i+1}/{epochs} - Loss: {loss.item():.4f}")


Epoch 25/300 - Loss: 0.2665
Epoch 50/300 - Loss: 0.2059
Epoch 75/300 - Loss: 0.4026
Epoch 100/300 - Loss: 0.3718
Epoch 125/300 - Loss: 0.2378
Epoch 150/300 - Loss: 0.3081
Epoch 175/300 - Loss: 0.4294
Epoch 200/300 - Loss: 0.2898
Epoch 225/300 - Loss: 0.2249
Epoch 250/300 - Loss: 0.1683
Epoch 275/300 - Loss: 0.2701
Epoch 300/300 - Loss: 0.2726



## Evaluation


In [13]:

model.eval()

with torch.no_grad():

    y_val = model(cat_test, cont_test)

    loss = criterion(y_val, y_test)

    predicted = torch.argmax(y_val, dim=1)

    accuracy = (predicted == y_test).sum().item() / len(y_test)

print(f"Test Loss: {loss.item():.4f}")
print(f"Test Accuracy: {accuracy*100:.2f}%")


Test Loss: 0.2568
Test Accuracy: 88.54%



## BONUS: Prediction Function
This function allows users to input new data and get predictions.


In [14]:

# Category mappings
category_maps = {
    col: dict(enumerate(df[col].cat.categories))
    for col in cat_cols
}

reverse_maps = {
    col: {v:k for k,v in category_maps[col].items()}
    for col in cat_cols
}


In [15]:

def predict_income(age,
                   sex,
                   education,
                   education_num,
                   marital_status,
                   workclass,
                   occupation,
                   hours_per_week):

    model.eval()

    cat_data = torch.tensor([[
        reverse_maps['sex'][sex],
        reverse_maps['education'][education],
        reverse_maps['marital-status'][marital_status],
        reverse_maps['workclass'][workclass],
        reverse_maps['occupation'][occupation]
    ]], dtype=torch.int64)

    cont_data = torch.tensor([[
        age,
        education_num,
        hours_per_week
    ]], dtype=torch.float32)

    with torch.no_grad():
        prediction = model(cat_data, cont_data)
        pred_class = torch.argmax(prediction, dim=1).item()

    if pred_class == 1:
        return "Income > $50K"
    else:
        return "Income <= $50K"


In [16]:

# Example prediction
predict_income(
    age=35,
    sex='Male',
    education='Bachelors',
    education_num=13,
    marital_status='Married',
    workclass='Private',
    occupation='Exec-managerial',
    hours_per_week=45
)


'Income > $50K'